# Occluded datasets: synthetic occlusions of the clean faces, one Kaggle dataset per source

Run this notebook **once per dataset** (ExpW, AffectNet, RAF-DB), with only that dataset's inputs:

| Dataset | Inputs | Output → Kaggle dataset |
|---|---|---|
| ExpW | `clean-faces`, `expw-crops` | `occluded-expw` |
| AffectNet | `clean-faces`, `affectnet_1` | `occluded-affectnet` |
| RAF-DB | `clean-faces-rafdb`, `rafdb-crops` | `occluded-rafdb` |

Settings: **Accelerator None** (CPU only), **Internet on** (to clone the code).

Every clean face gets random occlusions (mouth / eyes / brows / nose, random shapes, sizes and colours) drawn on its
256x256 crop with the clean landmarks, so occluded and clean images are pixel-aligned (`src/utils/occlusion.py`).
Train faces get 2 variants, val / test faces 1. The output is a few big files (`images.bin`, `masks.bin`,
`occluded_meta.csv` with the occluded face area of every image, `preview/`), read by `src/utils/occluded_store.py`.

## 1. Code

In [ ]:
import os, sys, glob
os.chdir('/kaggle/working')
!rm -rf /tmp/smirk && git clone -q --depth 1 -b occlusion-aware https://github.com/THUONGVIP09/smirk.git /tmp/smirk
!cd /tmp/smirk && git log --oneline -1
sys.path.insert(0, '/tmp/smirk')

## 2. Find the dataset to occlude

In [ ]:
CLEAN_DIRS = sorted(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/clean_meta.csv', recursive=True))
IMAGE_DIRS = {}
for folder in glob.glob('/kaggle/input/**/images', recursive=True):
    path = os.path.relpath(folder, '/kaggle/input').lower()
    name = next((k for k in ('expw', 'affectnet', 'rafdb') if k in path), None)
    if name and os.path.isdir(folder):
        IMAGE_DIRS[name] = folder
        print(f'{name}: {folder} ({len(os.listdir(folder))} images)')
print('clean dirs:', CLEAN_DIRS)
assert CLEAN_DIRS, 'add clean-faces (or clean-faces-rafdb) as input'
assert len(IMAGE_DIRS) == 1, f'add the crops of exactly one dataset as input (found {list(IMAGE_DIRS)})'
DATASET, IMAGE_DIR = next(iter(IMAGE_DIRS.items()))
OUT = f'/kaggle/working/occluded_{DATASET}'
print('dataset:', DATASET, '→', OUT)

## 3. Quick check on 200 faces

In [ ]:
CLEAN_ARGS = ' '.join(f'--clean_dir "{d}"' for d in CLEAN_DIRS)
!python /tmp/smirk/datasets/preprocess_scripts/make_occluded_dataset.py {CLEAN_ARGS} --image_dir "{IMAGE_DIR}" --dataset {DATASET} --out_dir /tmp/occluded_check --limit 200

## 4. All clean faces (~10-30 min on CPU)

In [ ]:
!python /tmp/smirk/datasets/preprocess_scripts/make_occluded_dataset.py {CLEAN_ARGS} --image_dir "{IMAGE_DIR}" --dataset {DATASET} --out_dir {OUT} --num_processes 4
!ls -lh {OUT}

## 5. Look at some occluded faces and at the occluded area

In [ ]:
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt
from src.utils.occluded_store import OccludedStore

store = OccludedStore(OUT)
meta = pd.read_csv(f'{OUT}/occluded_meta.csv', dtype={'name': str})
print(pd.crosstab(meta['expression'], meta['split'], margins=True))
print('occluded face area by split:'); print(meta.groupby('split')['occluded_area'].describe().round(3))

sample = meta.sample(12, random_state=0)
fig, axes = plt.subplots(3, 12, figsize=(24, 6.5))
for i, (_, r) in enumerate(sample.iterrows()):
    clean = cv2.imread(f"{IMAGE_DIR}/{r['name']}.png")[..., ::-1]
    occluded, mask = store.get(DATASET, r['name'], int(r['variant']))
    for ax, picture in zip(axes[:, i], [clean, occluded, mask]):
        ax.imshow(picture, cmap='gray' if picture.ndim == 2 else None); ax.axis('off')
    axes[1, i].set_title(f"{r['parts']}\narea={r['occluded_area']:.2f}", fontsize=7)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 3))
plt.hist(meta['occluded_area'], bins=40, color='#2a78d6')
plt.xlabel('occluded fraction of the face'); plt.ylabel('images'); plt.title('Occluded face area', loc='left')
plt.gca().spines[['top', 'right']].set_visible(False); plt.show()

## 6. Save

*Save Version → Save & Run All* (**Accelerator None**), then on the finished version: *Output → New Dataset*,
named `occluded-expw` / `occluded-affectnet` / `occluded-rafdb`, **Private**.